In [1]:
import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
)
from scipy.special import expit

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)

/home/kobat/bbs-with-AI-monitoring/.env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
DATA_PATH = "../data/processed/toxic_dataset_multilabel.csv"
MODEL_NAME = "cl-tohoku/bert-base-japanese-v3"
OUTPUT_DIR = "./models/toxic-bert-multilabel"

CATEGORIES = [
    "卑語",
    "差別",
    "迷惑行為",
    "猥褻",
    "出会い・プライバシー侵害",
    "違法行為",
    "偏向表現",
]

MAX_LENGTH = 128
THRESHOLD = 0.5
RANDOM_STATE = 42

print("カテゴリ数:", len(CATEGORIES))
print("モデル:", MODEL_NAME)

カテゴリ数: 7
モデル: cl-tohoku/bert-base-japanese-v3


In [3]:
df = pd.read_csv(DATA_PATH)

# 投稿文が空の行を除外
df = df.dropna(subset=["text"]).copy()
df["text"] = df["text"].astype(str)

# ラベルを0/1の整数型にする
df[CATEGORIES] = df[CATEGORIES].astype(int)

print("データ件数:", len(df))
print("列名:", df.columns.tolist())

display(df[["text"] + CATEGORIES].head())

print("\nカテゴリ別の該当件数")
display(df[CATEGORIES].sum().sort_values(ascending=False))

print(
    "全カテゴリが0の投稿数:",
    (df[CATEGORIES].sum(axis=1) == 0).sum()
)

データ件数: 429
列名: ['id', 'text', 'level_score', 'level', 'classification', 'category', 'annotation_num', '卑語', '差別', '迷惑行為', '猥褻', '出会い・プライバシー侵害', '違法行為', '偏向表現']


,text,卑語,差別,迷惑行為,猥褻,出会い・プライバシー侵害,違法行為,偏向表現
0,いわゆる天使だろ,0,0,0,0,0,0,0
1,底知れないカリスマ性があるよな,0,0,0,0,0,0,0
2,ありがとう,0,0,0,0,0,0,0
3,地味にこれが一番きつい,0,0,0,0,0,0,0
4,ヤクザより怖い,0,1,0,0,0,0,0



カテゴリ別の該当件数


迷惑行為            64
差別              30
偏向表現            17
猥褻              11
卑語               8
違法行為             3
出会い・プライバシー侵害     1
dtype: int64

全カテゴリが0の投稿数: 319


In [4]:
train_df, temp_df = train_test_split(
    df,
    test_size=0.2,
    random_state=RANDOM_STATE,
)

valid_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=RANDOM_STATE,
)

print("学習データ:", len(train_df))
print("検証データ:", len(valid_df))
print("テストデータ:", len(test_df))

print("\n各データのカテゴリ別件数")

for name, data in [
    ("学習", train_df),
    ("検証", valid_df),
    ("テスト", test_df),
]:
    print(f"\n【{name}データ】")
    display(data[CATEGORIES].sum())

学習データ: 343
検証データ: 43
テストデータ: 43

各データのカテゴリ別件数

【学習データ】


卑語               6
差別              24
迷惑行為            54
猥褻               9
出会い・プライバシー侵害     1
違法行為             3
偏向表現            14
dtype: int64


【検証データ】


卑語              2
差別              6
迷惑行為            8
猥褻              2
出会い・プライバシー侵害    0
違法行為            0
偏向表現            2
dtype: int64


【テストデータ】


卑語              0
差別              0
迷惑行為            2
猥褻              0
出会い・プライバシー侵害    0
違法行為            0
偏向表現            1
dtype: int64

In [5]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


def tokenize_texts(texts):
    return tokenizer(
        texts,
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
    )


sample = tokenize_texts(["これは動作確認用の文章です。"])

print("入力文章:", "これは動作確認用の文章です。")
print("トークンID:", sample["input_ids"][0][:20])
print("Attention mask:", sample["attention_mask"][0][:20])

入力文章: これは動作確認用の文章です。
トークンID: [2, 12538, 465, 15763, 13576, 3908, 464, 18530, 13037, 385, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Attention mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0]
